In [ ]:
import os
import time
from typing import Dict, Any, Optional

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    plt = None
    HAS_MPL = False

# OpenAI 环境配置
USE_OPENAI = bool(os.getenv("OPENAI_API_KEY"))
DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

if USE_OPENAI:
    try:
        from openai import OpenAI
        openai_client: Optional[OpenAI] = OpenAI()
    except Exception:
        openai_client = None
        USE_OPENAI = False
else:
    openai_client = None

class SimpleLLM:
    def __init__(self, model_name: str = "dummy-model"):
        self.model_name = model_name
        self.total_tokens_used = 0
        self.total_requests = 0

    def count_tokens(self, text: str) -> int:
        return len(text.split())

    def generate(self, prompt: str) -> str:
        if USE_OPENAI and openai_client is not None:
            try:
                resp = openai_client.chat.completions.create(
                    model=DEFAULT_MODEL,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.7,
                    max_tokens=800,
                )
                return resp.choices[0].message.content or ""
            except Exception as e:
                return f"[OpenAI 调用失败：{e}]"
        tokens = self.count_tokens(prompt)
        self.total_tokens_used += tokens
        self.total_requests += 1
        return f"[未检测到 OPENAI_API_KEY，返回占位响应。你的提示约计 {tokens} 个 token。]"

    def get_stats(self) -> Dict[str, Any]:
        return {
            "total_tokens": self.total_tokens_used,
            "total_requests": self.total_requests,
            "avg_tokens_per_request": self.total_tokens_used / max(1, self.total_requests),
        }

llm = SimpleLLM()


In [ ]:
atomic_prompt = "为一家三口（爸爸、妈妈、4 岁小女孩）规划 3 天的日照海边亲子旅游行程。"
print("原子提示：", atomic_prompt)
print("Token 数：", llm.count_tokens(atomic_prompt))
print("生成响应…\n")
response = llm.generate(atomic_prompt)
print(response)


In [ ]:
prompts = [
    "为一家三口（爸爸、妈妈、4 岁小女孩）规划 3 天的日照海边亲子旅游行程。",
    "为一家三口（爸爸、妈妈、4 岁小女孩）规划 3 天的日照海边行程，按每日上午/下午/晚间分段安排，并标注路程与时长。",
    "为一家三口规划 3 天日照亲子行：适合 4 岁幼儿；避开高强度活动与长时间排队；包含餐饮与午休；提供交通方式与人均预算区间（总预算 3000–5000 元）。",
]

results = []
for i, p in enumerate(prompts, 1):
    t0 = time.time()
    r = llm.generate(p)
    t1 = time.time()
    results.append({
        "prompt": p,
        "tokens": llm.count_tokens(p),
        "response": r,
        "latency": t1 - t0,
    })
    print(f"\n提示 {i}：{p}")
    print("Token 数：", results[-1]["tokens"])
    print("延迟：{:.3f}s".format(results[-1]["latency"]))
    print("响应：\n", r)


In [ ]:
quality_scores = [3, 6, 8]
if HAS_MPL:
    plt.figure(figsize=(8, 5))
    xs = [r["tokens"] for r in results]
    plt.plot(xs, quality_scores, marker='o')
    plt.xlabel('提示 Token 数')
    plt.ylabel('输出质量（占位 1-10）')
    plt.title('Token–质量 ROI（示意）')
    plt.grid(True)
    for i, (x, y) in enumerate(zip(xs, quality_scores), 1):
        plt.annotate(f"提示 {i}", (x, y), textcoords="offset points", xytext=(0, 8), ha='center')
    plt.show()
else:
    print("[未安装 matplotlib，跳过绘图]")


In [ ]:
enhanced_prompt = """任务：为一家三口（爸爸、妈妈、4 岁小女孩）规划 3 天的日照海边亲子旅游行程。

目标与边界：
- 行程应松弛有度，适合 4 岁幼儿（避免过度奔波、过度日晒与高强度项目）
- 每天按 上午 / 下午 / 晚间 分段列出安排，并注明预计时长
- 覆盖：亲子友好景点、沙滩玩耍、餐饮与午休、室内备选（遇到下雨/高温）
- 说明地面交通（步行/公交/打车/替代建议）
- 预算：总预算 3000–5000 元，给出人均预算区间与主要花费构成
- 附加：为 4 岁幼儿提供安全与防晒注意事项、简明备品清单

请用结构化小标题与有序列表输出，便于家长直接照单执行。"""

print("增强提示 Token 数：", llm.count_tokens(enhanced_prompt))
print("生成响应…\n")
print(llm.generate(enhanced_prompt))


In [ ]:
def measure_consistency(prompt: str, n_samples: int = 3) -> Dict[str, Any]:
    rs = []
    for _ in range(n_samples):
        rs.append(llm.generate(prompt))
    # 占位一致性分（真实可替换为语义相似度平均值）
    return {
        "prompt": prompt,
        "responses": rs,
        "consistency_score": 0.5,
    }

basic_consistency = measure_consistency(prompts[0])
enhanced_consistency = measure_consistency(enhanced_prompt)

print("基础提示一致性（占位）：", basic_consistency["consistency_score"])
print("增强提示一致性（占位）：", enhanced_consistency["consistency_score"])
